# AIOps Autonomous Network Guardian & Predictive Simulation
## Part 1: Multivariate Telemetry Simulation & Time-Series Lag Features

In [2]:
import numpy as np
import pandas as pd

# Generate 30-day multivariate telemetry across multiple South African regional towers
np.random.seed(42)
timestamps = pd.date_range(start="2026-09-01 00:00:00", periods=720, freq="h")
node_ids = [
    "Tower_JHB_Gauteng_01",
    "Tower_JHB_Gauteng_02",
    "Tower_DBN_KZN_01",
    "Tower_POT_NWP_5",
    "Tower_CPT_WC_03",
    "Tower_DUR_KZN_07",
    "Tower_MHK_NWP_02",
    "Tower_PRE_Gauteng_04",
]
data_frames = []

for node in node_ids:
  n = len(timestamps)
  latency = np.random.normal(loc=18.5, scale=2.5, size=n)
  cpu_load = np.random.normal(loc=42.0, scale=6.0, size=n)
  prb_util = np.random.normal(loc=58.0, scale=7.5, size=n)
  call_drop_rate = np.random.normal(loc=0.7, scale=0.2, size=n)
  rsrp = np.random.normal(loc=-92.0, scale=4.0, size=n)
  cabinet_temp = np.random.normal(loc=30.0, scale=3.0, size=n)
  power_status = np.ones(n, dtype=int)
  mos_score = np.random.normal(loc=4.5, scale=0.2, size=n)
  trouble_tickets = np.random.poisson(lam=1.2, size=n)

  # Inject continuous failure episodes (3 to 6 hours)
  num_episodes = 12
  for _ in range(num_episodes):
    start_idx = np.random.randint(10, n - 15)
    duration = np.random.randint(3, 7)

    for idx in range(start_idx, start_idx + duration):
      cabinet_temp[idx] += np.random.uniform(15.0, 25.0)
      latency[idx] += np.random.uniform(50.0, 90.0)
      cpu_load[idx] = np.random.uniform(88.0, 99.0)
      prb_util[idx] = np.random.uniform(93.0, 99.0)
      call_drop_rate[idx] += np.random.uniform(3.5, 6.0)
      power_status[idx] = 0
      mos_score[idx] = np.random.uniform(1.5, 2.8)
      trouble_tickets[idx] += np.random.randint(15, 35)

  # Network Health Index Calculation
  health_score = np.full(n, 100.0)
  health_score -= cpu_load * 0.15
  health_score -= np.where(
      cabinet_temp > 40, (cabinet_temp - 40) * 1.5, 0
  )
  health_score -= np.where(power_status == 0, 30.0, 0.0)
  health_score -= call_drop_rate * 5.0
  health_score -= np.where(mos_score < 3.0, (3.0 - mos_score) * 10, 0)
  health_score = np.clip(np.round(health_score, 2), 0.0, 100.0)

  HEALTH_THRESHOLD = 65.0
  anomaly_mask = np.where(health_score < HEALTH_THRESHOLD, 1, 0)
  icasa_violation = np.where(call_drop_rate > 3.0, 1, 0)

  df_node = pd.DataFrame({
      "Timestamp": timestamps,
      "Node_ID": node,
      "Network_Health_Score": health_score,
      "Latency_ms": np.round(latency, 2),
      "CPU_Load_Pct": np.clip(np.round(cpu_load, 2), 0, 100),
      "PRB_Utilization_Pct": np.clip(np.round(prb_util, 2), 0, 100),
      "Cabinet_Temp_C": np.round(cabinet_temp, 2),
      "Power_Grid_Stable": power_status,
      "Call_Drop_Rate_Pct": np.clip(np.round(call_drop_rate, 2), 0, 100),
      "ICASA_Violation": icasa_violation,
      "MOS_Score": np.clip(np.round(mos_score, 2), 1.0, 5.0),
      "Customer_Tickets": trouble_tickets,
      "Is_Anomalous": anomaly_mask,
  })
  data_frames.append(df_node)

master_health_df = pd.concat(data_frames, ignore_index=True)

# Build Predictive Lag Features
df_list = []
for node, group in master_health_df.groupby("Node_ID"):
  group = group.copy()
  group["Temp_Lag_1"] = group["Cabinet_Temp_C"].shift(1)
  group["Temp_Lag_2"] = group["Cabinet_Temp_C"].shift(2)
  group["Health_Lag_1"] = group["Network_Health_Score"].shift(1)
  group["CPU_Lag_1"] = group["CPU_Load_Pct"].shift(1)
  group["Future_Anomaly"] = group["Is_Anomalous"].shift(-1)
  df_list.append(group)

predictive_df = pd.concat(df_list).dropna()
print(
    f"Simulation complete. Total feature rows ready for modeling:"
    f" {len(predictive_df)}"
)

Simulation complete. Total feature rows ready for modeling: 5736


## Part 2: Supervised Machine Learning Model (Random Forest Classifier)

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

feature_cols = [
    "Network_Health_Score",
    "Health_Lag_1",
    "Cabinet_Temp_C",
    "Temp_Lag_1",
    "Temp_Lag_2",
    "CPU_Load_Pct",
    "CPU_Lag_1",
    "Power_Grid_Stable",
    "Call_Drop_Rate_Pct",
]

X = predictive_df[feature_cols]
y = predictive_df["Future_Anomaly"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest with balanced class weights
predictive_model = RandomForestClassifier(
    n_estimators=100, random_state=42, class_weight="balanced"
)
predictive_model.fit(X_train, y_train)

y_pred = predictive_model.predict(X_test)

print("=== Episode-Based Predictive Model Performance ===")
print(f"Prediction Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, zero_division=0))

# Run batch predictions across the dataset
predictive_df["Model_Prediction"] = predictive_model.predict(
    predictive_df[feature_cols]
)

=== Episode-Based Predictive Model Performance ===
Prediction Accuracy: 96.86%

=== Classification Report ===
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.98      1064
         1.0       0.85      0.69      0.76        84

    accuracy                           0.97      1148
   macro avg       0.91      0.84      0.87      1148
weighted avg       0.97      0.97      0.97      1148



## Part 3: Closed-Loop Automation Playbooks & Batch Execution

In [6]:
def execute_self_healing_playbook(row):
  """Evaluates telemetry when a future anomaly is predicted

  and triggers the corresponding automated remediation playbook.
  """
  print(
      f"\n[ALERT] Predictive AIOps Warning for {row['Node_ID']} at"
      f" {row['Timestamp']}!"
  )
  print(
      "-> Model forecasts a network failure within the next hour (Health"
      f" Score: {row['Network_Health_Score']})"
  )

  actions_taken = []

  # 1. Check for Power Grid Failure (Load Shedding / Grid Trip)
  if row["Power_Grid_Stable"] == 0:
    actions_taken.append(
        "[Power Playbook] Grid failure detected. Engaging backup lithium"
        " battery reserve & notifying diesel generator dispatch."
    )

  # 2. Check for Thermal Stress (HVAC / Overheating)
  if row["Cabinet_Temp_C"] > 40.0:
    actions_taken.append(
        f"[Thermal Playbook] Cabinet temp is {row['Cabinet_Temp_C']}°C."
        " Overriding HVAC to maximum cooling capacity."
    )

  # 3. Check for Radio Congestion
  if row["PRB_Utilization_Pct"] > 90.0:
    actions_taken.append(
        "[Traffic Playbook] High PRB utilization ("
        f"{row['PRB_Utilization_Pct']}%). Executing SDN load balancing to"
        " offload traffic to adjacent tower."
    )

  # Default fallback if multi-factor stress is minor
  if not actions_taken:
    actions_taken.append(
        "[General Playbook] Initiating soft-reset of control-plane signaling"
        " stack to clear memory leaks."
    )

  print("=== Automated Remediation Executed ===")
  for action in actions_taken:
    print(action)
  print("-" * 60)


# Execute batch scan across all towers
network_warnings = predictive_df[predictive_df["Model_Prediction"] == 1]

print(
    f"=== AIOps Network Scan Complete ==="
    f"\nTotal Towers Monitored: {predictive_df['Node_ID'].nunique()}"
    f"\nImminent Failures Flagged: {len(network_warnings)}\n"
)

# Loop through warnings and trigger playbooks
for _, warning_row in network_warnings.iterrows():
  execute_self_healing_playbook(warning_row)

# Optional: Export for Power BI dashboard
predictive_df.to_csv("satnac_aiops_dashboard_data.csv", index=False)
print(
    "\n[Dashboard] Data successfully exported to 'satnac_aiops_dashboard_data.csv'"
)

=== AIOps Network Scan Complete ===
Total Towers Monitored: 8
Imminent Failures Flagged: 403


[ALERT] Predictive AIOps Warning for Tower_CPT_WC_03 at 2026-09-02 11:00:00!
-> Model forecasts a network failure within the next hour (Health Score: 91.49)
=== Automated Remediation Executed ===
[General Playbook] Initiating soft-reset of control-plane signaling stack to clear memory leaks.
------------------------------------------------------------

[ALERT] Predictive AIOps Warning for Tower_CPT_WC_03 at 2026-09-02 12:00:00!
-> Model forecasts a network failure within the next hour (Health Score: 7.52)
=== Automated Remediation Executed ===
[Power Playbook] Grid failure detected. Engaging backup lithium battery reserve & notifying diesel generator dispatch.
[Thermal Playbook] Cabinet temp is 51.08°C. Overriding HVAC to maximum cooling capacity.
[Traffic Playbook] High PRB utilization (98.13%). Executing SDN load balancing to offload traffic to adjacent tower.
------------------------------